In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Analisis_Scraping") \
    .config("spark.mongodb.input.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.mongodb.output.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()

df = spark.read.format("mongo").load()

df.printSchema()
df.show(5)

root
 |-- _id: struct (nullable = true)
 |    |-- oid: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- identificador: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+-------------------+----------------+--------------------+-----+
|                 _id|      fecha_captura|           grupo|       identificador|valor|
+--------------------+-------------------+----------------+--------------------+-----+
|{69cd130fd75f49e1...|2026-04-01 12:43:56|Nombre del grupo|A Light in the Attic|51.77|
|{69cd130fd75f49e1...|2026-04-01 12:43:56|Nombre del grupo|  Tipping the Velvet|53.74|
|{69cd130fd75f49e1...|2026-04-01 12:43:56|Nombre del grupo|          Soumission| 50.1|
|{69cd130fd75f49e1...|2026-04-01 12:43:56|Nombre del grupo|       Sharp Objects|47.82|
|{69cd130fd75f49e1...|2026-04-01 12:43:56|Nombre del grupo|Sapiens: A Brief ...|54.23|
+--------------------+-------------------+----------------+

In [2]:
from pyspark.sql.functions import col

df_filtrado = df.filter(
    (col("identificador").isNotNull()) &
    (col("identificador") != "") &
    (col("valor").isNotNull()) &
    (col("valor") > 0)
)

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 120
Registros validos: 120


In [3]:
from pyspark.sql.functions import col, trim, to_timestamp

# LIMPIEZA COMPLETA
df_clean = df \
    .drop("_id") \
    .filter(
        col("identificador").isNotNull() &
        col("valor").isNotNull() &
        (col("identificador") != "") &
        (col("grupo") != "") &
        (col("valor") > 0)
    ) \
    .dropDuplicates(["identificador", "valor"]) \
    .withColumn("identificador", trim(col("identificador"))) \
    .withColumn("grupo", trim(col("grupo"))) \
    .withColumn(
        "fecha_captura",
        to_timestamp(col("fecha_captura"), "yyyy-MM-dd HH:mm:ss")
    )

# VALIDACIÓN FINAL
print("Registros originales:", df.count())
print("Registros limpios:", df_clean.count())

df_clean.printSchema()
df_clean.show(5)

Registros originales: 120
Registros limpios: 60
root
 |-- fecha_captura: timestamp (nullable = true)
 |-- grupo: string (nullable = true)
 |-- identificador: string (nullable = true)
 |-- valor: double (nullable = true)

+-------------------+----------------+--------------------+-----+
|      fecha_captura|           grupo|       identificador|valor|
+-------------------+----------------+--------------------+-----+
|2026-04-01 12:43:58|Nombre del grupo|#HigherSelfie: Wa...|23.11|
|2026-04-01 12:43:56|Nombre del grupo|A Light in the Attic|51.77|
|2026-04-01 12:43:57|Nombre del grupo|Aladdin and His W...|53.13|
|2026-04-01 12:43:57|Nombre del grupo|America's Cradle ...| 22.5|
|2026-04-01 12:43:57|Nombre del grupo| Behind Closed Doors|52.22|
+-------------------+----------------+--------------------+-----+
only showing top 5 rows



In [4]:
from pyspark.sql import functions as F

reporte_grupo = df_clean.groupBy("grupo").agg(
    F.count("identificador").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE DATOS POR GRUPO:")
reporte_grupo.show()

ANALISIS DE DATOS POR GRUPO:
+----------------+---------------+-----------------+-------------+-------------+
|           grupo|cantidad_libros|  precio_promedio|precio_minimo|precio_maximo|
+----------------+---------------+-----------------+-------------+-------------+
|Nombre del grupo|             60|35.00266666666667|        12.84|        57.31|
+----------------+---------------+-----------------+-------------+-------------+



In [9]:
ticket_salida = df_clean.groupBy("grupo").agg(
    F.count("identificador").alias("total_libros"),
    F.round(F.avg("valor"), 2).alias("precio_medio"),
    F.min("valor").alias("precio_min"),
    F.max("valor").alias("precio_max"),
    F.max("fecha_captura").alias("ultima_sincronizacion")
)

ticket_salida.show()

+----------------+------------+------------+----------+----------+---------------------+
|           grupo|total_libros|precio_medio|precio_min|precio_max|ultima_sincronizacion|
+----------------+------------+------------+----------+----------+---------------------+
|Nombre del grupo|          60|        35.0|     12.84|     57.31|  2026-04-01 12:43:58|
+----------------+------------+------------+----------+----------+---------------------+

